# Day Trader Metrics

No API keys needed in this, no trades are actioned, only an analysis of public market data.

1. Data sourced live using `CCXT` API library, no scraping required.
2. Indicators derived using `pandas-ta-classic`, replacing `numpy.2`



### Environment Setup

Run this cell each time kernel starts, re-running builds the same environment, loads interpreter into notebook, and set renderers from using package list in `inputs/requirements.txt`. Before running this setup chunk, make sure to log into your Binance and Alpaca accounts, advized to use the prferred protocols for Binance.US below here: https://docs.ccxt.com/en/latest/exchange-markets.html


In [11]:
import sys, subprocess
from pathlib import Path

REQUIREMENTS = Path("inputs/requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",   "--break-system-packages", 
    "--disable-pip-version-check", "-r", str(REQUIREMENTS)], check=True,)

import warnings, numpy as np, pandas as pd
from datetime import datetime
import ccxt
import pandas_ta_classic as ta
from pykalman import KalmanFilter
import plotly, plotly.graph_objects as go, plotly.io as pio
warnings.filterwarnings("ignore")
pio.renderers.default = "plotly_mimetype+notebook_connected"  # renders inline in Positron

print(f"Environment ready  ·  python {sys.version.split()[0]}")
print(f"  interpreter : {sys.executable}")
print(f"  ccxt {ccxt.__version__} · pandas {pd.__version__} · numpy {np.__version__} · plotly {plotly.__version__}")

Environment ready  ·  python 3.11.13
  interpreter : /Users/seamus/.local/share/uv/python/cpython-3.11.13-macos-aarch64-none/bin/python
  ccxt 4.5.59 · pandas 2.3.3 · numpy 2.4.6 · plotly 6.8.0


### Import Data

Live price data is sourced between 24 hours and 1 week from multiple trading platforms. Five data filters were added to sampling controls below, including `EXCHANGE`, `SANDBOX`, `TIMEFRAME`, `LIMIT`, and `TOP_N`, as defined below

In [12]:
# ---- Data Filters ----
EXCHANGE  = "binance"   # data source: "binance", "binanceus", "kraken", "coinbase", ...
SANDBOX   = False       # True = testnet / fake money (mainly for the trading step later)
TIMEFRAME = "1d"        # candle size: "1m","5m","1h","4h","1d","1w"
LIMIT     = 300         # candles pulled per coin (max 1000)
TOP_N     = 20          # how many of the most-traded /USDT pairs to scan

exchange = getattr(ccxt, EXCHANGE)()
exchange.set_sandbox_mode(SANDBOX)
print(f"Configured: {EXCHANGE} | sandbox={SANDBOX} | {TIMEFRAME} candles x{LIMIT} | top {TOP_N}")

Configured: binance | sandbox=False | 1d candles x300 | top 20


In [13]:
def calculate_indicator(symbol, timeframe=TIMEFRAME, limit=LIMIT):
    bars = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
    df = pd.DataFrame(bars[:-1], columns=["timestamp","open","high","low","close","volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df.set_index("timestamp")
    close, low = df["close"].iloc[-1], df["low"].iloc[-1]

    # Kalman filter: a random-walk smoother of the close.
    kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1],
                      initial_state_mean=0, initial_state_covariance=1,
                      observation_covariance=1, transition_covariance=.01)
    state_means, _ = kf.filter(df["close"].values)
    df["kf_mean"] = state_means
    kalman = df["kf_mean"].iloc[-1]
    above_kalman = bool(low > kalman)

    # Trend: EMA-14 leading the Kalman mean.
    df.ta.ema(length=14, append=True)
    ema_cross = bool(df["EMA_14"].iloc[-1] > kalman)

    # Bollinger(14) mean-reversion envelope.
    bb = df.ta.bbands(length=14)
    bbl, bbu = bb["BBL_14_2.0"].iloc[-1], bb["BBU_14_2.0"].iloc[-1]

    # Ichimoku forward spans (context), computed once.
    ich = df.ta.ichimoku()[1]
    isa_9, isb_26 = ich["ISA_9"].iloc[-1], ich["ISB_26"].iloc[-1]

    # Archer MA trend flag, RSI, Choppiness.
    amat = bool(df.ta.amat()["AMATe_LR_8_21_2"].iloc[-1] == 1)
    rsi = float(df.ta.rsi().iloc[-1])
    chop = round(float(df.ta.chop().iloc[-1]), 2)
    
    # Candle shape: doji / dragonfly / gravestone
    o, h, l, c = df["open"].iloc[-1], df["high"].iloc[-1], df["low"].iloc[-1], df["close"].iloc[-1]
    rng   = h - l
    body  = abs(c - o)
    upper = h - max(o, c)
    lower = min(o, c) - l
    doji       = bool(rng > 0 and body <= 0.10 * rng)
    dragonfly  = bool(doji and lower >= 0.6 * rng and upper <= 0.10 * rng)
    gravestone = bool(doji and upper >= 0.6 * rng and lower <= 0.10 * rng)

    # MACD (12,26,9)
    macd_line   = df["close"].ewm(span=12, adjust=False).mean() - df["close"].ewm(span=26, adjust=False).mean()
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    macd, sig           = macd_line.iloc[-1], signal_line.iloc[-1]
    macd_prev, sig_prev = macd_line.iloc[-2], signal_line.iloc[-2]
    macd_buy  = bool(macd_prev <= sig_prev and macd > sig)
    macd_sell = bool(macd_prev >= sig_prev and macd < sig)
    macd_below_zero = bool(macd < 0)
    buy  = amat and ema_cross and above_kalman
    sell = (not amat) and (not ema_cross) and (not above_kalman)    
    row = dict(Symbol=symbol, Buy=buy, Sell=sell, Close=round(float(close), 4),
               RSI=round(rsi, 2), Chop=chop, AMAT=amat,
               Ichimoku_9=round(float(isa_9), 4), Ichimoku_26=round(float(isb_26), 4),
               EMA_gt_Kalman=ema_cross, Low_gt_Kalman=above_kalman,
               Doji=doji, Dragonfly=dragonfly, Gravestone=gravestone,
               MACD=round(float(macd),4), MACD_Signal=round(float(sig),4),
               MACD_Buy=macd_buy, MACD_Sell=macd_sell, MACD_below_zero=macd_below_zero)
               
    return df, row

def plot(symbol):
    df, _ = calculate_indicator(symbol)
    fig = go.Figure(go.Candlestick(x=df.index, open=df.open, high=df.high, low=df.low, close=df.close, name=symbol))
    fig.add_trace(go.Scatter(x=df.index, y=df["kf_mean"], name="Kalman", line=dict(color="orange", width=2), opacity=0.7))
    fig.add_trace(go.Scatter(x=df.index, y=df["EMA_14"], name="EMA-14", line=dict(color="purple", width=2), opacity=0.7))
    fig.update_layout(title=symbol, xaxis_rangeslider_visible=False)
    return fig

def color_boolean(val):
    if val is True:  return "background-color: lightgreen"
    if val is False: return "background-color: pink"
    return "background-color: lightblue"

print("Engine ready.")

Engine ready.


### Analyze Data

The following pulls 300 daily candles and computes the following:

- Kalman Filter: a price smoothing noisy fluctuations, key indicator of close of day price strength
- Bollinger Bands: An elastic envelope derived from 14-day price average. Narrowing suggests quiet market, flaring suggest volatile.
- Ichimoku Spans: Two mean price projections used as trend map, suggesting uptrend or downtrends between 9 adn 26 days.
- AMAT: Archer Moving Average Trends used as binary indicators showing diverging fast and slow averages, or aligning showing sustained trend (1 = "trending up").
- Relative Strength Index (RSI). Speedometer, above 70, price running fast risks overselling and then cooling off. Below 30 may mean due to bounce.
- Choppiness Index: High values suggest inertia, low suggest clean trend.

Key values in these metrics in sell and buy point variables above, copied here again for review: `ema_crossover = ema_14 > ema_91` 


The originals scanned every Binance.US ticker, which is slow and noisy. Here we take the most liquid spot **/USDT** pairs by quote volume. Raise `TOP_N` to widen the scan; each extra symbol adds roughly a second.

In [14]:
tickers = exchange.fetch_tickers()
pairs = [(s, d.get('quoteVolume') or 0) for s, d in tickers.items()
         if s.endswith('/USDT') and ':' not in s]
universe = [s for s, _ in sorted(pairs, key=lambda x: -x[1])[:TOP_N]]
today = datetime.now().strftime('%Y-%m-%d')
print(f"{today}: scanning {len(universe)} pairs")
universe

2026-06-18: scanning 20 pairs


['USDC/USDT',
 'BTC/USDT',
 'ETH/USDT',
 'WLD/USDT',
 'USD1/USDT',
 'SOL/USDT',
 'ZEC/USDT',
 'XRP/USDT',
 'BNB/USDT',
 'NEAR/USDT',
 'XLM/USDT',
 'ASTER/USDT',
 'MEGA/USDT',
 'RLUSD/USDT',
 'FDUSD/USDT',
 'SPCXB/USDT',
 'XPL/USDT',
 'RE/USDT',
 'UNI/USDT',
 'SUI/USDT']

### Score

Run the engine over the universe into one table. The `try/except` skips symbols too new to have full indicator history. For example a token listed days ago has no Ichimoku cloud and skips these values instead of hiding them. 

In [15]:
rows = []
for symbol in universe:
    try: rows.append(calculate_indicator(symbol)[1])
    except Exception as e: print(f"skip {symbol}: {type(e).__name__}")
results = pd.DataFrame(rows)
print(f"\nscored {len(results)} pairs | {int(results.Buy.sum())} buys | {int(results.Sell.sum())} sells")
results.head()

skip MEGA/USDT: TypeError
skip SPCXB/USDT: KeyError
skip RE/USDT: IndexError

scored 17 pairs | 4 buys | 1 sells


,Symbol,Buy,Sell,Close,RSI,Chop,AMAT,Ichimoku_9,Ichimoku_26,EMA_gt_Kalman,Low_gt_Kalman,Doji,Dragonfly,Gravestone,MACD,MACD_Signal,MACD_Buy,MACD_Sell,MACD_below_zero
0,USDC/USDT,False,False,1.0006,52.78,17.31,True,1.0080,1.0108,False,False,False,False,False,-0.0000,-0.0000,False,False,True
1,BTC/USDT,False,False,64509.4000,38.74,54.00,True,66314.5150,70990.4550,False,False,False,False,False,-2449.8918,-3053.8891,False,False,True
2,ETH/USDT,False,False,1750.6000,41.55,51.49,True,1777.1525,1964.7100,False,False,False,False,False,-87.2995,-112.4537,False,False,True
3,WLD/USDT,True,False,0.6587,68.50,54.27,True,0.5336,0.4748,True,True,False,False,False,0.0745,0.0597,False,False,False
4,USD1/USDT,True,False,1.0008,62.70,63.37,True,1.0001,1.0000,True,True,False,False,False,0.0002,0.0001,False,False,False


## Trading Plan

### Entry Points

Buy trends are displayed first with Kalman flags rendered, then lowest RSI at the top. Sells are shown as the mirror, which the originals' automation checklist is built upon.

In [16]:
top = (results.sort_values(['Buy','EMA_gt_Kalman','AMAT','RSI'], ascending=[False,False,False,True]).reset_index(drop=True).head(10))
bottom = (results.sort_values(['Sell','AMAT','RSI'],ascending=[False,False,False]).reset_index(drop=True).head(10))
print("ranked.")

if top['Buy'].any():
    print("Entry candidates showing bullish trends and Kalman lead prices:")
    for s in top.loc[top['Buy'], 'Symbol']: print(" ", s)
else: print("No long signals showing strongest-ranked names.")

top.style.map(color_boolean)

ranked.
Entry candidates showing bullish trends and Kalman lead prices:
  XLM/USDT
  USD1/USDT
  XPL/USDT
  WLD/USDT


,Symbol,Buy,Sell,Close,RSI,Chop,AMAT,Ichimoku_9,Ichimoku_26,EMA_gt_Kalman,Low_gt_Kalman,Doji,Dragonfly,Gravestone,MACD,MACD_Signal,MACD_Buy,MACD_Sell,MACD_below_zero
0,XLM/USDT,True,False,0.225600,61.690000,55.770000,True,0.210500,0.218900,True,True,False,False,False,0.007700,0.007300,True,False,False
1,USD1/USDT,True,False,1.000800,62.700000,63.370000,True,1.000100,1.000000,True,True,False,False,False,0.000200,0.000100,False,False,False
2,XPL/USDT,True,False,0.112700,67.440000,39.660000,True,0.091700,0.091700,True,True,False,False,False,0.001600,-0.002400,False,False,False
3,WLD/USDT,True,False,0.658700,68.500000,54.270000,True,0.533600,0.474800,True,True,False,False,False,0.074500,0.059700,False,False,False
4,NEAR/USDT,False,False,2.182000,51.310000,50.320000,True,2.348000,2.166000,True,False,False,False,False,0.054100,0.068300,False,False,False
5,BTC/USDT,False,False,64509.400000,38.740000,54.000000,True,66314.515000,70990.455000,False,False,False,False,False,-2449.891800,-3053.889100,False,False,True
6,SUI/USDT,False,False,0.767300,39.140000,56.070000,True,0.826700,1.042000,False,False,False,False,False,-0.050700,-0.061300,False,False,True
7,ETH/USDT,False,False,1750.600000,41.550000,51.490000,True,1777.152500,1964.710000,False,False,False,False,False,-87.299500,-112.453700,False,False,True
8,XRP/USDT,False,False,1.186800,44.710000,49.170000,True,1.202100,1.299900,False,False,False,False,False,-0.037700,-0.051000,False,False,True
9,SOL/USDT,False,False,72.050000,46.200000,50.960000,True,71.515000,79.270000,False,False,False,False,False,-2.914300,-4.120800,False,False,True


A chart showing strongest buy candidate, with the Kalman and EMA-14 trend lines:

In [17]:
from IPython.display import HTML
HTML(plot(top['Symbol'].iloc[0]).to_html(include_plotlyjs="cdn", full_html=False))

In [18]:
# Save the chart as a standalone, self-contained HTML file you can open in any browser.
# include_plotlyjs=True embeds the library so it works offline. No nbformat / .show() needed.
fig = plot(top['Symbol'].iloc[0])
fig.write_html('outputs/chart.html', include_plotlyjs=True, auto_open=False)
print('saved outputs/chart.html')


saved outputs/chart.html


### Exit Points

In [19]:
if bottom['Sell'].any():
    print("Exit candidates trending down showing prices below Kalman:")
    for s in bottom.loc[bottom['Sell'], 'Symbol']: print(" ", s)
else: print("No short signals today; showing weakest-ranked names.")
bottom.style.map(color_boolean)

Exit candidates trending down showing prices below Kalman:
  BNB/USDT


,Symbol,Buy,Sell,Close,RSI,Chop,AMAT,Ichimoku_9,Ichimoku_26,EMA_gt_Kalman,Low_gt_Kalman,Doji,Dragonfly,Gravestone,MACD,MACD_Signal,MACD_Buy,MACD_Sell,MACD_below_zero
0,BNB/USDT,False,True,601.630000,43.410000,55.320000,False,628.945000,651.100000,False,False,False,False,False,-10.923900,-11.429000,False,False,True
1,WLD/USDT,True,False,0.658700,68.500000,54.270000,True,0.533600,0.474800,True,True,False,False,False,0.074500,0.059700,False,False,False
2,XPL/USDT,True,False,0.112700,67.440000,39.660000,True,0.091700,0.091700,True,True,False,False,False,0.001600,-0.002400,False,False,False
3,ASTER/USDT,False,False,0.717000,63.550000,36.000000,True,0.698500,0.695500,False,False,False,False,False,-0.002900,-0.009300,False,False,True
4,USD1/USDT,True,False,1.000800,62.700000,63.370000,True,1.000100,1.000000,True,True,False,False,False,0.000200,0.000100,False,False,False
5,UNI/USDT,False,False,3.227000,61.760000,30.810000,True,3.035000,3.243000,False,True,False,False,False,-0.069400,-0.170400,False,False,True
6,XLM/USDT,True,False,0.225600,61.690000,55.770000,True,0.210500,0.218900,True,True,False,False,False,0.007700,0.007300,True,False,False
7,FDUSD/USDT,False,False,0.999000,56.980000,45.300000,True,0.998300,0.998000,False,True,False,False,False,-0.000000,-0.000100,False,False,True
8,USDC/USDT,False,False,1.000600,52.780000,17.310000,True,1.008000,1.010800,False,False,False,False,False,-0.000000,-0.000000,False,False,True
9,RLUSD/USDT,False,False,1.000800,52.230000,66.980000,True,1.001000,1.001000,False,False,False,False,False,-0.000000,-0.000000,False,False,True


Save data as csv file in `/outputs/` subfolder with datestamp for comparative analysis.

In [20]:
stamp = datetime.now().strftime('%Y%m%d')
path = f'./outputs/DailySignals_{stamp}.csv'
results.to_csv(path, index=False)
print("saved", path)

saved ./outputs/DailySignals_20260618.csv


### Decision Checklist

1. Read balances (USDT, BTC, ...).
2. Size each trade as cash divided by the number of long candidates.
3. Act on exits (`bottom`) before entries (`top`).

#### Render Reports using Quarto

 - `quarto preview day-metrics.ipynb --no-execute`
 - `quarto render day-metrics.ipynb --to html --no-execute`